# Importing the necessary libraries

In [ ]:
import pandas as pd
import numpy as np
import optuna
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# Loading the datasets

In [ ]:
blr_df = pd.read_csv('../Data/Processed/blr_df.csv')
hyd_df = pd.read_csv('../Data/Processed/hyd_df.csv')
pune_df = pd.read_csv('../Data/Processed/pune_df.csv')

## Converting the Date Column to Datetime Column

In [ ]:
blr_df['Date'] = pd.to_datetime(blr_df['Date'])
hyd_df['Date'] = pd.to_datetime(hyd_df['Date'])
pune_df['Date'] = pd.to_datetime(pune_df['Date'])

In [ ]:
blr_df = blr_df.sort_values('Date')
hyd_df = hyd_df.sort_values('Date')
pune_df = pune_df.sort_values('Date')

# Feature and Target Columns

In [ ]:
features = ['AP', 'DPT', 'WS', 'WSD', 'RH']
target = 'LST'

In [ ]:
blr_X = blr_df[features].values
blr_y = blr_df[target].values

In [ ]:
hyd_X = hyd_df[features].values
hyd_y = hyd_df[target].values

In [ ]:
pune_X = pune_df[features].values
pune_y = pune_df[target].values

# Scaling the Features

In [ ]:
scaler = StandardScaler()

In [ ]:
blr_X_scaled = scaler.fit_transform(blr_X)
hyd_X_scaled = scaler.transform(hyd_X)
pune_X_scaled = scaler.transform(pune_X)

# Creating Sequences

In [ ]:
def create_sequences(X, y, seq_length=30):
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_length):
        X_seq.append(X[i:i+seq_length])
        y_seq.append(y[i+seq_length])
    return np.array(X_seq), np.array(y_seq)

In [ ]:
seq_len = 30


In [ ]:
blr_X_seq, blr_y_seq = create_sequences(blr_X_scaled, blr_y, seq_len)
hyd_X_seq, hyd_y_seq = create_sequences(hyd_X_scaled, hyd_y, seq_len)
pune_X_seq, pune_y_seq = create_sequences(pune_X_scaled, pune_y, seq_len)

# 80-10-10 Split

In [ ]:
def split_data(X_seq, y_seq):
    train_idx = int(0.8 * len(X_seq))
    val_idx = int(0.9 * len(X_seq))
    
    X_train = X_seq[:train_idx]
    y_train = y_seq[:train_idx]
    X_val = X_seq[train_idx:val_idx]
    y_val = y_seq[train_idx:val_idx]
    X_test = X_seq[val_idx:]
    y_test = y_seq[val_idx:]
    
    return X_train, y_train, X_val, y_val, X_test, y_test

In [ ]:
blr_X_train, blr_y_train, blr_X_val, blr_y_val, blr_X_test, blr_y_test = split_data(blr_X_seq, blr_y_seq)
hyd_X_train, hyd_y_train, hyd_X_val, hyd_y_val, hyd_X_test, hyd_y_test = split_data(hyd_X_seq, hyd_y_seq)
pune_X_train, pune_y_train, pune_X_val, pune_y_val, pune_X_test, pune_y_test = split_data(pune_X_seq, pune_y_seq)

# Converting to PyTorch Tensors

In [ ]:
def convert_to_tensor(X_train, y_train, X_val, y_val, X_test, y_test):
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).to(device)
    X_val_tensor = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_val_tensor = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1).to(device)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1).to(device)
    return X_train_tensor, y_train_tensor, X_val_tensor, y_val_tensor, X_test_tensor, y_test_tensor

In [ ]:
blr_X_train_tensor, blr_y_train_tensor, blr_X_val_tensor, blr_y_val_tensor, blr_X_test_tensor, blr_y_test_tensor = convert_to_tensor(blr_X_train, blr_y_train, blr_X_val, blr_y_val, blr_X_test, blr_y_test)
hyd_X_train_tensor, hyd_y_train_tensor, hyd_X_val_tensor, hyd_y_val_tensor, hyd_X_test_tensor, hyd_y_test_tensor = convert_to_tensor(hyd_X_train, hyd_y_train, hyd_X_val, hyd_y_val, hyd_X_test, hyd_y_test)
pune_X_train_tensor, pune_y_train_tensor, pune_X_val_tensor, pune_y_val_tensor, pune_X_test_tensor, pune_y_test_tensor = convert_to_tensor(pune_X_train, pune_y_train, pune_X_val, pune_y_val, pune_X_test, pune_y_test)

# Defining the LSTM Model

In [ ]:
class BiLSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, dropout=dropout, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_size * 2, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out

# Objective Function for Optuna with Early Stopping

In [ ]:
def compute_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    # NSE = 1 - (sum((obs - sim)^2) / sum((obs - mean(obs))^2))
    nse = 1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - np.mean(y_true)) ** 2)

    # RSR = RMSE / std_dev_obs
    rmse = np.sqrt(mse)
    rsr = rmse / np.std(y_true)

    # PBIAS = 100 * sum(sim - obs) / sum(obs)
    pbias = 100 * np.sum(y_pred - y_true) / np.sum(y_true)

    return mse, mae, r2, nse, rsr, pbias

In [ ]:
def objective(trial, X_train_tensor, y_train_tensor, X_val_tensor, y_val_tensor, X_test_tensor, y_test_tensor, model_save_path): 
    hidden_size = trial.suggest_int("hidden_size", 50, 500)
    num_layers = trial.suggest_int("num_layers", 1, 2)
    dropout = trial.suggest_float("dropout", 0.0, 0.4) if num_layers > 1 else 0.0
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [2, 4, 8, 16, 32, 64])
    epochs = trial.suggest_int("epochs", 50, 50)
    patience = 5

    model = BiLSTMModel(X_train_tensor.shape[2], hidden_size, num_layers, dropout).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    dataset = torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor)
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

    best_val_loss = float('inf')
    best_model_state = None
    epochs_no_improve = 0

    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_preds = model(X_val_tensor.to(device))
            val_loss = criterion(val_preds, y_val_tensor.to(device)).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict()
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            break

    if best_model_state is not None:
        torch.save(best_model_state, model_save_path)

    model.load_state_dict(best_model_state)
    model.eval()
    with torch.no_grad():
        test_preds = model(X_test_tensor).cpu().numpy()
        test_y = y_test_tensor.cpu().numpy()
    
    mse, mae, r2, nse, rsr, pbias = compute_metrics(test_y, test_preds)
    print(f"Test MSE: {mse:.4f}, MAE: {mae:.4f}, R²: {r2:.4f}, NSE: {nse:.4f}, RSR: {rsr:.4f}, PBIAS: {pbias:.4f}")

    return mse

# Running Optuna for Each City

In [ ]:
os.makedirs('../Models/BiLSTM/', exist_ok=True)

In [ ]:
blr_study = optuna.create_study(direction="minimize")
blr_study.optimize(lambda trial: objective(trial, blr_X_train_tensor, blr_y_train_tensor, blr_X_val_tensor, blr_y_val_tensor, blr_X_test_tensor, blr_y_test_tensor, "../Models/BiLSTM/blr_lstm_model.pth"), n_trials=30)

In [ ]:
hyd_study = optuna.create_study(direction="minimize")
hyd_study.optimize(lambda trial: objective(trial, hyd_X_train_tensor, hyd_y_train_tensor, hyd_X_val_tensor, hyd_y_val_tensor, hyd_X_test_tensor, hyd_y_test_tensor, "../Models/BiLSTM/hyd_lstm_model.pth"), n_trials=30)

In [ ]:
pune_study = optuna.create_study(direction="minimize")
pune_study.optimize(lambda trial: objective(trial, pune_X_train_tensor, pune_y_train_tensor, pune_X_val_tensor, pune_y_val_tensor, pune_X_test_tensor, pune_y_test_tensor, "../Models/BiLSTM/pune_lstm_model.pth"), n_trials=30)